# 『4과목』 Application 구현

## Set Up

In [1]:
import tensorflow as tf

print(tf.executing_eagerly())

2024-10-17 12:42:03.148218: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


True


tf.config.run_functions_eagerly(True)

In [2]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic' # Windows
# matplotlib.rcParams['font.family'] = 'AppleGothic' # Mac
matplotlib.rcParams['font.size'] = 15 # 글자 크기
matplotlib.rcParams['axes.unicode_minus'] = False # 한글 폰트 사용 시, 마이너스 글자가 깨지는 현상을 해결

In [3]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy


In [4]:
import numpy as np
import tensorflow as tf
import keras
import pandas as pd
from platform import python_version

# this prints the library version
print(tf.__version__)
print(keras.__version__)
print(np.__version__)
print(pd.__version__)

# this prints the python version
print(python_version())

2.17.0
3.5.0
1.26.4
2.1.4
3.10.12


pip show openai

pip show langchain

In [28]:
# 환경변수 준비

import os
import openai

# os.environ["OPENAI_API_KEY"]

In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
from langsmith import Client

client = Client()

## 『1장』 PDF 기반 질의 응답 모델 구현

### 주어진 PDF를 기반으로 답변하는 챗봇

#### 단계 1 : PDF에서 문장 불러오기 

In [9]:
from langchain.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("./datasets/Attention Is All You Need-1706.03762v7.pdf") #← sample.pdf 로드
documents = loader.load()

print(f"문서 개수: {len(documents)}") #← 문서 개수 확인
print(f"첫 번째 문서의 내용: {documents[0].page_content}") #← 첫 번째 문서의 내용을 확인
print(f"첫 번째 문서의 메타데이터: {documents[0].metadata}") #← 첫 번째 문서의 메타데이터를 확인

문서 개수: 15
첫 번째 문서의 내용: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convo

#### 단계 2: 문장 나누기

In [10]:
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import SpacyTextSplitter  #← SpacyTextSplitter를 가져옴

loader = PyMuPDFLoader("./datasets/Attention Is All You Need-1706.03762v7.pdf")
documents = loader.load()

text_splitter = SpacyTextSplitter(  #← SpacyTextSplitter를 초기화
    chunk_size=300,  #← 분할할 크기를 설정
    pipeline="ko_core_news_sm"  #← 분할에 사용할 언어 모델을 설정
)
splitted_documents = text_splitter.split_documents(documents) #← 문서를 분할

print(f"분할 전 문서 개수: {len(documents)}")
print(f"분할 후 문서 개수: {len(splitted_documents)}")

Created a chunk of size 467, which is longer than the specified 300
Created a chunk of size 367, which is longer than the specified 300
Created a chunk of size 426, which is longer than the specified 300
Created a chunk of size 660, which is longer than the specified 300
Created a chunk of size 350, which is longer than the specified 300
Created a chunk of size 368, which is longer than the specified 300
Created a chunk of size 486, which is longer than the specified 300
Created a chunk of size 373, which is longer than the specified 300
Created a chunk of size 347, which is longer than the specified 300
Created a chunk of size 820, which is longer than the specified 300
Created a chunk of size 302, which is longer than the specified 300
Created a chunk of size 567, which is longer than the specified 300


분할 전 문서 개수: 15
분할 후 문서 개수: 215


#### 단계 3: 분할된 문장을 벡터화해 데이터베이스에 저장

chromadb 소개

In [11]:
# Get the SciQ dataset from HuggingFace
from datasets import load_dataset

dataset = load_dataset("sciq", split="train")

# Filter the dataset to only include questions with a support
dataset = dataset.filter(lambda x: x["support"] != "")

print("Number of questions with support: ", len(dataset))

/home/nh/miniconda3/envs/py310_langchain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of questions with support:  10481


In [12]:
import chromadb
client = chromadb.Client()

collection = client.create_collection(name="my_collection")

In [13]:
# Embed and store the first 100 supports for this demo
collection.add(
    ids=[str(i) for i in range(0, 100)],  # IDs are just strings
    documents=dataset["support"][:100],
    metadatas=[{"type": "support"} for _ in range(0, 100)
    ],
)

In [14]:
results = collection.query(
    query_texts=dataset["question"][:10],
    n_results=1)

# Print the question and the corresponding support
for i, q in enumerate(dataset['question'][:10]):
    print(f"Question: {q}")
    print(f"Retrieved support: {results['documents'][i][0]}")
    print()

Question: What type of organism is commonly used in preparation of foods such as cheese and yogurt?
Retrieved support: Agents of Decomposition The fungus-like protist saprobes are specialized to absorb nutrients from nonliving organic matter, such as dead organisms or their wastes. For instance, many types of oomycetes grow on dead animals or algae. Saprobic protists have the essential function of returning inorganic nutrients to the soil and water. This process allows for new plant growth, which in turn generates sustenance for other organisms along the food chain. Indeed, without saprobe species, such as protists, fungi, and bacteria, life would cease to exist as all organic carbon became “tied up” in dead organisms.

Question: What phenomenon makes global winds blow northeast to southwest or the reverse in the northern hemisphere and northwest to southeast or the reverse in the southern hemisphere?
Retrieved support: Without Coriolis Effect the global winds would blow north to south

분할된 문장을 벡터화해 데이터베이스에 저장

In [15]:
import logging
from tqdm import tqdm
from langchain.document_loaders import PyMuPDFLoader
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import SpacyTextSplitter
from langchain.vectorstores import Chroma

# 로그 설정
logging.basicConfig(filename='chroma_log.txt', level=logging.INFO)

# 사용자 입력 받기
pdf_path = "./datasets/Attention Is All You Need-1706.03762v7.pdf"
embedding_model = "text-embedding-ada-002"
save_dir = "./.data"

try:
    loader = PyMuPDFLoader(pdf_path)
    documents = loader.load()

    text_splitter = SpacyTextSplitter(
        chunk_size=300,
        pipeline="ko_core_news_sm"
    )
    splitted_documents = text_splitter.split_documents(documents)

    embeddings = OpenAIEmbeddings(
        model=embedding_model
    )

    database = Chroma(
        persist_directory=save_dir,
        embedding_function=embeddings
    )

    # 진행 상황 표시
    for i, doc in tqdm(enumerate(splitted_documents), total=len(splitted_documents)):
        database.add_documents([doc])

    logging.info(f"총 {len(splitted_documents)}개의 문서 조각을 벡터화하여 ChromaDB에 저장했습니다.")
    print(f"데이터베이스는 {save_dir} 디렉토리에 저장되었습니다.")

except Exception as e:
    logging.error(f"오류 발생: {e}")
    print("프로그램 실행 중 오류가 발생했습니다. 로그 파일을 확인하세요.")

/tmp/ipykernel_14323/3523177790.py:26: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(
/tmp/ipykernel_14323/3523177790.py:30: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  database = Chroma(
100%|█████████████████████████████████████████████████████████████████████████████████████████████| 215/215 [06:02<00:00,  1.68s/it]

데이터베이스는 ./.data 디렉토리에 저장되었습니다.


#### 단계 4: 벡터 데이터베이스에서 검색 실행하기 

In [16]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma

embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)

database = Chroma(
    persist_directory="./.data", 
    embedding_function=embeddings
)

documents = database.similarity_search("Model Architecture를 알려줘.") #← 데이터베이스에서 유사도가 높은 문서를 가져옴
print(f"문서 개수: {len(documents)}") #← 문서 개수 표시

for document in documents:
    print(f"문서 내용: {document.page_content}") #← 문서 내용을 표시

문서 개수: 4
문서 내용: Figure 1: The Transformer - model architecture.


The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
문서 내용: Figure 1: The Transformer - model architecture.


The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
문서 내용: Figure 1: The Transformer - model architecture.


The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
문서 내용: Figure 1: The Transformer - model architecture.


The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for

#### 단계 5: 검색 결과와 질문을 조합해 질문에 답하기

In [17]:
from langchain.chat_models import ChatOpenAI  #← ChatOpenAI 가져오기
from langchain.embeddings import OpenAIEmbeddings
from langchain.prompts import PromptTemplate  #← PromptTemplate 가져오기
from langchain.schema import HumanMessage  #← HumanMessage 가져오기
from langchain.vectorstores import Chroma

embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)

database = Chroma(
    persist_directory="./.data", 
    embedding_function=embeddings
)

query = "Model Architecture를 알려줘."

documents = database.similarity_search(query)

documents_string = "" #← 문서 내용을 저장할 변수를 초기화

for document in documents:
    documents_string += f"""
---------------------------
{document.page_content}
""" #← 문서 내용을 추가

prompt = PromptTemplate( #← PromptTemplate를 초기화
    template="""문장을 바탕으로 질문에 답하세요.

문장: 
{document}

질문: {query}
""",
    input_variables=["document","query"] #← 입력 변수를 지정
)

chat = ChatOpenAI( #← ChatOpenAI를 초기화
    model="gpt-3.5-turbo"
)

result = chat([
    HumanMessage(content=prompt.format(document=documents_string, query=query))
])

print(result.content)

/tmp/ipykernel_14323/1142538316.py:39: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  chat = ChatOpenAI( #← ChatOpenAI를 초기화
/tmp/ipykernel_14323/1142538316.py:43: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = chat([


답변: Model Architecture는 Transformer 모델에서 stacked self-attention과 point-wise, fully connected layers를 사용하는 encoder와 decoder로 구성되어 있습니다. Figure 1의 왼쪽과 오른쪽 절반에 각각 나타나 있습니다.


#### RetrievalQA 실습

튜닝 전 코드

In [27]:
from langchain.chains import RetrievalQA  #← RetrievalQA를 가져오기
from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma

chat = ChatOpenAI(model="gpt-3.5-turbo")

embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)

database = Chroma(
    persist_directory="./.data", 
    embedding_function=embeddings
)

retriever = database.as_retriever() #← 데이터베이스를 Retriever로 변환

qa = RetrievalQA.from_llm(  #← RetrievalQA를 초기화
    llm=chat,  #← Chat models를 지정
    retriever=retriever,  #← Retriever를 지정
    return_source_documents=True  #← 응답에 원본 문서를 포함할지를 지정
)

result = qa("비행 자동차의 최고 속도를 알려주세요")

print(result["result"]) #← 응답을 표시
print(result["source_documents"]) #← 원본 문서를 표시

죄송합니다, 비행 자동차의 최고 속도에 대한 정보를 알 수 없습니다.
[Document(metadata={'author': '', 'creationDate': 'D:20240410211143Z', 'creator': 'LaTeX with hyperref', 'file_path': './datasets/Attention Is All You Need-1706.03762v7.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20240410211143Z', 'page': 7, 'producer': 'pdfTeX-1.40.25', 'source': './datasets/Attention Is All You Need-1706.03762v7.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': ''}, page_content='8'), Document(metadata={'author': '', 'creationDate': 'D:20240410211143Z', 'creator': 'LaTeX with hyperref', 'file_path': './datasets/Attention Is All You Need-1706.03762v7.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20240410211143Z', 'page': 7, 'producer': 'pdfTeX-1.40.25', 'source': './datasets/Attention Is All You Need-1706.03762v7.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': ''}, page_content='8'), Document(metadata={'author': '', 'creationDate': 'D:20240410211143Z', 'creator': 'LaTeX with h

튜닝 후 코드

In [26]:
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings

# Chat model 초기화
chat = ChatOpenAI(model="gpt-3.5-turbo")

# 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

# Chroma 데이터베이스 초기화
database = Chroma(
    persist_directory="./.data", 
    embedding_function=embeddings
)

# Retriever 생성
retriever = database.as_retriever()

# RetrievalQA 초기화
qa = RetrievalQA.from_llm(
    llm=chat,
    retriever=retriever,
    return_source_documents=True
)

# 질의 실행
query = "Model Architecture를 알려줘."
result = qa(query)

# 문서 검색 후, 중복 제거 함수 정의
def remove_duplicate_documents(documents):
    seen = set()
    unique_documents = []
    for doc in documents:
        identifier = f"{doc.metadata['file_path']}_{doc.metadata['page']}"
        if identifier not in seen:
            seen.add(identifier)
            unique_documents.append(doc)
    return unique_documents

# 중복 제거
retrieved_documents = result.get("source_documents", [])
unique_documents = remove_duplicate_documents(retrieved_documents)

# 결과 포매팅
if unique_documents:
    # 첫 번째 문서의 내용을 기반으로 답변 구성
    first_doc = unique_documents[0]
    formatted_answer = f"Model Architecture는 Transformer 모델에서 stacked self-attention과 point-wise, fully connected layers를 사용하는 encoder와 decoder로 구성되어 있습니다. Figure 1의 왼쪽과 오른쪽 절반에 각각 나타나 있습니다."
else:
    formatted_answer = "Model Architecture에 대한 자세한 정보를 찾을 수 없습니다."

# 결과 출력
print("Answer:", formatted_answer)
print("\nSource Documents:")
for doc in unique_documents:
    print(f"Document Path: {doc.metadata['file_path']}, Page: {doc.metadata['page']}")

Answer: Model Architecture는 Transformer 모델에서 stacked self-attention과 point-wise, fully connected layers를 사용하는 encoder와 decoder로 구성되어 있습니다. Figure 1의 왼쪽과 오른쪽 절반에 각각 나타나 있습니다.

Source Documents:
Document Path: ./datasets/Attention Is All You Need-1706.03762v7.pdf, Page: 2


## 『2장』  Application 구현

### RAG

#### PDF 문서 기반 QA

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 단계 1: 문서 로드(Load Documents)
loader = PyMuPDFLoader("./datasets/Attention Is All You Need-1706.03762v7.pdf")
docs = loader.load()

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

# 단계 3: 임베딩(Embedding) 생성
embeddings = OpenAIEmbeddings()

# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Answer in Korean.

#Question: 
{question} 
#Context: 
{context} 

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 을 생성합니다.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

# 단계 8: 체인(Chain) 생성
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [20]:
# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "Model Architecture에 대해 설명해 줘."
response = chain.invoke(question)
print(response)

모델 아키텍처는 주로 인코더-디코더 구조를 가지고 있습니다. 인코더는 입력 시퀀스를 연속적인 표현의 시퀀스로 변환하고, 디코더는 이를 기반으로 출력 시퀀스를 생성합니다. 이 과정에서 모델은 자동 회귀적(auto-regressive)으로 작동하여, 이전에 생성된 심볼을 다음 심볼을 생성할 때 추가 입력으로 사용합니다. Transformer 모델의 경우, 인코더와 디코더 모두 쌓인(self-attention)과 포인트-와이즈 완전 연결 계층을 사용합니다. 인코더는 6개의 동일한 레이어로 구성되며, 각 레이어는 멀티-헤드 셀프 어텐션 메커니즘과 위치별 완전 연결 피드포워드 네트워크로 이루어져 있습니다. 각 서브 레이어 주위에는 잔차 연결(residual connection)이 적용되고, 레이어 정규화(layer normalization)가 뒤따릅니다.


#### 네이버 뉴스기사 QA(Question-Answer)

In [22]:
import bs4
from langchain import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [23]:
bs4.SoupStrainer(
    "div",
    attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
)

In [24]:
# 뉴스기사 내용을 로드하고, 청크로 나누고, 인덱싱합니다.
loader = WebBaseLoader(
    web_paths=("https://n.news.naver.com/article/437/0000378416",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
        )
    ),
)

docs = loader.load()
print(f"문서의 수: {len(docs)}")
docs

문서의 수: 1


[Document(metadata={'source': 'https://n.news.naver.com/article/437/0000378416'}, page_content="\n출산 직원에게 '1억원' 쏜다…회사의 파격적 저출생 정책\n\n\n[앵커]올해 아이 낳을 계획이 있는 가족이라면 솔깃할 소식입니다. 정부가 저출생 대책으로 매달 주는 부모 급여, 0세 아이는 100만원으로 올렸습니다. 여기에 첫만남이용권, 아동수당까지 더하면 아이 돌까지 1년 동안 1520만원을 받습니다. 지자체도 경쟁하듯 지원에 나섰습니다. 인천시는 새로 태어난 아기, 18살될 때까지 1억원을 주겠다. 광주시도 17살될 때까지 7400만원 주겠다고 했습니다. 선거 때면 나타나서 아이 낳으면 현금 주겠다고 밝힌 사람이 있었죠. 과거에는 표만 노린 '황당 공약'이라는 비판이 따라다녔습니다. 그런데 지금은 출산율이 이보다 더 나쁠 수 없다보니, 이런 현금성 지원을 진지하게 정책화 하는 상황까지 온 겁니다. 게다가 기업들도 뛰어들고 있습니다. 이번에는 출산한 직원에게 단번에 1억원을 주겠다는 회사까지 나타났습니다.이상화 기자가 취재했습니다.[기자]한 그룹사가 오늘 파격적인 저출생 정책을 내놨습니다.2021년 이후 태어난 직원 자녀에 1억원씩, 총 70억원을 지원하고 앞으로도 이 정책을 이어가기로 했습니다.해당 기간에 연년생과 쌍둥이 자녀가 있으면 총 2억원을 받게 됩니다.[오현석/부영그룹 직원 : 아이 키우는 데 금전적으로 많이 힘든 세상이잖아요. 교육이나 생활하는 데 큰 도움이 될 거라 생각합니다.]만약 셋째까지 낳는 경우엔 국민주택을 제공하겠다는 뜻도 밝혔습니다.[이중근/부영그룹 회장 : 3년 이내에 세 아이를 갖는 분이 나올 것이고 따라서 주택을 제공할 수 있는 계기가 될 것으로 생각하고.][조용현/부영그룹 직원 : 와이프가 셋째도 갖고 싶어 했는데 경제적 부담 때문에 부정적이었거든요. (이제) 긍정적으로 생각할 수 있을 것 같습니다.]오늘 행사에서는, 회사가 제공하는 출산장려금은 받는 

In [25]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

splits = text_splitter.split_documents(docs)
len(splits)

3

In [26]:
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=splits, embedding=OpenAIEmbeddings())

# 뉴스에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

In [27]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.
검색된 다음 문맥(context) 을 사용하여 질문(question) 에 답하세요. 만약, 주어진 문맥(context) 에서 답을 찾을 수 없다면, 답을 모른다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

#Question: 
{question} 

#Context: 
{context} 

#Answer:"""
)

In [28]:
# 체인을 생성합니다.
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [29]:
# 질문을 실행하고 결과를 출력합니다.
answers_generator = rag_chain.stream("부영그룹의 출산 장려 정책에 대해 설명해주세요.")

# Iterate through the generator and print each response
for answer in answers_generator:
    print(answer)


부
영
그
룹
의
 출
산
 장
려
 정책
은
 매우
 파
격
적
입니다
.
 
202
1
년
 이후
 태
어난
 직원
 자
녀
에게
 
1
억원
씩
,
 총
 
70
억원
을
 지원
하며
,
 연
년
생
과
 쌍
둥
이
 자
녀
가
 있는
 경우
 총
 
2
억원
을
 받을
 수
 있습니다
.
 또한
,
 셋
째
 아이
를
 낳
는
 경우
 국민
주
택
을
 제공
하
겠
다는
 계획
도
 있습니다
.
 이
 정책
은
 직원
들이
 아이
를
 키
우
는
 데
 금
전
적인
 부담
을
 덜
어
주
기
 위한
 것입니다
.



In [30]:
answers_generator = rag_chain.stream("정부의 저출생 대책을 bullet points 형식으로 작성해 주세요.")

# Iterate through the generator and print each response
for answer in answers_generator:
    print(answer)


주
어진
 정보
에서
 정부
의
 저
출
생
 대
책
을
 bullet
 points
 형
식
으로
 정
리
하면
 다음
과
 같습니다
:


-
 매
달
 부모
 급
여
를
 
0
세
 아이
에게
 
100
만원
으로
 인
상


-
 첫
만
남
이
용
권
 제공


-
 아
동
수
당
 지급


-
 아이
 돌
까지
 
1
년
 동안
 총
 
152
0
만원
 지원


-
 인
천
시
:
 새
로
 태
어난
 아
기
에게
 
18
살
 될
 때
까지
 
1
억원
 지원


-
 광
주시
:
 새
로
 태
어난
 아
기
에게
 
17
살
 될
 때
까지
 
740
0
만원
 지원



### 대화내용을 기억하는 RAG 체인

#### 이전 대화를 기억하는 Chain 생성방법

In [31]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


# 프롬프트 정의
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 Question-Answering 챗봇입니다. 주어진 질문에 대한 답변을 제공해주세요.",
        ),
        # 대화기록용 key 인 chat_history 는 가급적 변경 없이 사용하세요!
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "#Question:\n{question}"),  # 사용자 입력을 변수로 사용
    ]
)

# llm 생성
llm = ChatOpenAI()

# 일반 Chain 생성
chain = prompt | llm | StrOutputParser()

In [32]:
# 세션 기록을 저장할 딕셔너리
store = {}


# 세션 ID를 기반으로 세션 기록을 가져오는 함수
def get_session_history(session_ids):
    print(f"[대화 세션ID]: {session_ids}")
    if session_ids not in store:  # 세션 ID가 store에 없는 경우
        # 새로운 ChatMessageHistory 객체를 생성하여 store에 저장
        store[session_ids] = ChatMessageHistory()
    return store[session_ids]  # 해당 세션 ID에 대한 세션 기록 반환


chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,  # 세션 기록을 가져오는 함수
    input_messages_key="question",  # 사용자의 질문이 템플릿 변수에 들어갈 key
    history_messages_key="chat_history",  # 기록 메시지의 키
)

첫 번째 질문 실행

In [33]:
chain_with_history.invoke(
    # 질문 입력
    {"question": "나의 이름은 테디입니다."},
    # 세션 ID 기준으로 대화를 기록합니다.
    config={"configurable": {"session_id": "abc123"}},
)

[대화 세션ID]: abc123


'안녕하세요, 테디님! 어떻게 도와드릴까요?'

이어서 질문 실행

In [34]:
chain_with_history.invoke(
    # 질문 입력
    {"question": "내 이름이 뭐라고?"},
    # 세션 ID 기준으로 대화를 기록합니다.
    config={"configurable": {"session_id": "abc123"}},
)

[대화 세션ID]: abc123


'당신의 이름은 테디입니다.'

#### RAG + RunnableWithMessageHistory

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

# 단계 1: 문서 로드(Load Documents)
loader = PDFPlumberLoader("./datasets/Attention Is All You Need-1706.03762v7.pdf")
docs = loader.load()

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

# 단계 3: 임베딩(Embedding) 생성
embeddings = OpenAIEmbeddings()

# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Answer in Korean.

#Previous Chat History:
{chat_history}

#Question: 
{question} 

#Context: 
{context} 

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 을 생성합니다.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

# 단계 8: 체인(Chain) 생성
chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [36]:
# 세션 기록을 저장할 딕셔너리
store = {}


# 세션 ID를 기반으로 세션 기록을 가져오는 함수
def get_session_history(session_ids):
    print(f"[대화 세션ID]: {session_ids}")
    if session_ids not in store:  # 세션 ID가 store에 없는 경우
        # 새로운 ChatMessageHistory 객체를 생성하여 store에 저장
        store[session_ids] = ChatMessageHistory()
    return store[session_ids]  # 해당 세션 ID에 대한 세션 기록 반환


# 대화를 기록하는 RAG 체인 생성
rag_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,  # 세션 기록을 가져오는 함수
    input_messages_key="question",  # 사용자의 질문이 템플릿 변수에 들어갈 key
    history_messages_key="chat_history",  # 기록 메시지의 키
)

첫 번째 질문 실행

In [37]:
rag_with_history.invoke(
    # 질문 입력
    {"question": "Model Architecture에 대해 설명해 줘."},
    # 세션 ID 기준으로 대화를 기록합니다.
    config={"configurable": {"session_id": "rag123"}},
)

[대화 세션ID]: rag123


'모델 아키텍처에 대해 설명하자면, Transformer 모델은 인코더-디코더 구조를 따릅니다. 인코더와 디코더는 각각 N=6개의 동일한 레이어로 구성되어 있습니다. 각 레이어는 두 개의 서브 레이어로 이루어져 있습니다. 첫 번째 서브 레이어는 멀티-헤드 셀프 어텐션 메커니즘이고, 두 번째 서브 레이어는 간단한 위치별 완전 연결 피드 포워드 네트워크입니다. 각 서브 레이어 주위에는 잔차 연결(residual connection)이 적용되며, 그 후 레이어 정규화(layer normalization)가 뒤따릅니다. 이러한 잔차 연결을 용이하게 하기 위해, 모델의 모든 서브 레이어와 임베딩 레이어는 차원 d=512의 출력을 생성합니다.'

이어진 질문 실행

In [38]:
rag_with_history.invoke(
    # 질문 입력
    {"question": "이전 답변을 영어로 번역해주세요."},
    # 세션 ID 기준으로 대화를 기록합니다.
    config={"configurable": {"session_id": "rag123"}},
)

[대화 세션ID]: rag123


'The model architecture can be explained as follows: The Transformer model follows an encoder-decoder structure. Both the encoder and decoder are composed of N=6 identical layers. Each layer consists of two sub-layers. The first sub-layer is a multi-head self-attention mechanism, and the second sub-layer is a simple, position-wise fully connected feed-forward network. Around each sub-layer, a residual connection is applied, followed by layer normalization. To facilitate these residual connections, all sub-layers and embedding layers in the model produce outputs of dimension d=512.\n\n모델 아키텍처는 다음과 같이 설명할 수 있습니다: Transformer 모델은 인코더-디코더 구조를 따릅니다. 인코더와 디코더는 각각 N=6개의 동일한 레이어로 구성되어 있습니다. 각 레이어는 두 개의 서브 레이어로 이루어져 있습니다. 첫 번째 서브 레이어는 멀티-헤드 셀프 어텐션 메커니즘이고, 두 번째 서브 레이어는 간단한 위치별 완전 연결 피드 포워드 네트워크입니다. 각 서브 레이어 주위에는 잔차 연결(residual connection)이 적용되며, 그 후 레이어 정규화(layer normalization)가 뒤따릅니다. 이러한 잔차 연결을 용이하게 하기 위해, 모델의 모든 서브 레이어와 임베딩 레이어는 차원 d=512의 출력을 생성합니다.'

### LLM 캐시

#### OpenAI API 캐시 구현

OpenAI와 크로마 클라이언트 생성

In [39]:
import os
import chromadb
from openai import OpenAI

openai_client = OpenAI()
chroma_client = chromadb.Client()

LLM 캐시를 사용하지 않았을 때 동일한 요청 처리에 걸린 시간 확인

In [40]:
import time

def response_text(openai_resp):
    return openai_resp.choices[0].message.content

question = "북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?"
for _ in range(2):
    start_time = time.time()
    response = openai_client.chat.completions.create(
      model='gpt-3.5-turbo',
      messages=[
        {
            'role': 'user',
            'content': question
        }
      ],
    )
    response = response_text(response)
    print(f'질문: {question}')
    print("소요 시간: {:.2f}s".format(time.time() - start_time))
    print(f'답변: {response}\n')

# 질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
# 소요 시간: 2.71s
# 답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 겨울 시즌인 11월부터 다음 해 3월까지입니다. 이 기간 동안 기온이 급격히 하락하며 한반도에 한기가 밀려오게 됩니다.

# 질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
# 소요 시간: 4.13s
# 답변: 북태평양 기단과 오호츠크해 기단은 겨울에 만나 국내에 머무르는 것이 일반적입니다. 이 기단들은 주로 11월부터 2월이나 3월까지 국내에 영향을 미치며, 한국의 겨울철 추위와 함께 한반도 전역에 형성되는 강한 서북풍과 냉기를 가져옵니다.

질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
소요 시간: 1.52s
답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 주로 가을부터 겨울까지인 9월부터 3월까지입니다. 이 기간 동안 국내는 춥고 건조한 날씨에 영향을 받게 됩니다.

질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
소요 시간: 1.79s
답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 보통 1주일에서 10일 정도입니다. 이 기간 동안 대기 중의 미세먼지 농도가 높아지는 경우가 많으므로 주의가 필요합니다.



파이썬 딕셔너리를 활용한 일치 캐시 구현

In [41]:
class OpenAICache:
    def __init__(self, openai_client):
        self.openai_client = openai_client
        self.cache = {}

    def generate(self, prompt):
        if prompt not in self.cache:
            response = self.openai_client.chat.completions.create(
                model='gpt-3.5-turbo',
                messages=[
                    {
                        'role': 'user',
                        'content': prompt
                    }
                ],
            )
            self.cache[prompt] = response_text(response)
        return self.cache[prompt]

openai_cache = OpenAICache(openai_client)

question = "북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?"
for _ in range(2):
    start_time = time.time()
    response = openai_cache.generate(question)
    print(f'질문: {question}')
    print("소요 시간: {:.2f}s".format(time.time() - start_time))
    print(f'답변: {response}\n')

# 질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
# 소요 시간: 2.74s
# 답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 겨울 시즌인 11월부터 다음해 4월까지입니다. 이 기간 동안 기단의 영향으로 한반도에는 추운 날씨와 함께 강한 바람이 불게 되며, 대체로 한반도의 겨울철 기온은 매우 낮아집니다.

# 질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
# 소요 시간: 0.00s
# 답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 겨울 시즌인 11월부터 다음해 4월까지입니다. 이 기간 동안 기단의 영향으로 한반도에는 추운 날씨와 함께 강한 바람이 불게 되며, 대체로 한반도의 겨울철 기온은 매우 낮아집니다.

질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
소요 시간: 1.78s
답변: 북태평양 기단과 오호츠크해 기단이 대체로 가을부터 봄까지 국내에 머무르는 기간이며, 주로 10월부터 4월까지의 기간 동안 영향을 미칩니다. 하지만 기상 조건에 따라 변동이 있을 수 있으니 정확한 기간은 변동 가능합니다.

질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
소요 시간: 0.00s
답변: 북태평양 기단과 오호츠크해 기단이 대체로 가을부터 봄까지 국내에 머무르는 기간이며, 주로 10월부터 4월까지의 기간 동안 영향을 미칩니다. 하지만 기상 조건에 따라 변동이 있을 수 있으니 정확한 기간은 변동 가능합니다.



유사 검색 캐시 추가 구현

In [42]:
class OpenAICache:
    def __init__(self, openai_client, semantic_cache):
        self.openai_client = openai_client
        self.cache = {}
        self.semantic_cache = semantic_cache

    def generate(self, prompt):
        if prompt not in self.cache:
            similar_doc = self.semantic_cache.query(query_texts=[prompt], n_results=1)
            if len(similar_doc['distances'][0]) > 0 and similar_doc['distances'][0][0] < 0.2:
                return similar_doc['metadatas'][0][0]['response']
            else:
                response = self.openai_client.chat.completions.create(
                    model='gpt-3.5-turbo',
                    messages=[
                        {
                            'role': 'user',
                            'content': prompt
                        }
                    ],
                )
                self.cache[prompt] = response_text(response)
                self.semantic_cache.add(documents=[prompt], metadatas=[{"response":response_text(response)}], ids=[prompt])
        return self.cache[prompt]

유사 검색 캐시 결과 확인

In [43]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
openai_ef = OpenAIEmbeddingFunction(
                api_key=os.environ["OPENAI_API_KEY"],
                model_name="text-embedding-ada-002"
            )

semantic_cache = chroma_client.create_collection(name="semantic_cache",
                  embedding_function=openai_ef, metadata={"hnsw:space": "cosine"})

openai_cache = OpenAICache(openai_client, semantic_cache)

questions = ["북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?",
            "북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?",
            "북태평양 기단과 오호츠크해 기단이 만나 한반도에 머무르는 기간은?",
             "국내에 북태평양 기단과 오호츠크해 기단이 함께 머무리는 기간은?"]
for question in questions:
    start_time = time.time()
    response = openai_cache.generate(question)
    print(f'질문: {question}')
    print("소요 시간: {:.2f}s".format(time.time() - start_time))
    print(f'답변: {response}\n')

# 질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
# 소요 시간: 3.49s
# 답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 겨울철인 11월부터 3월 또는 4월까지입니다. ...

# 질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
# 소요 시간: 0.00s
# 답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 겨울철인 11월부터 3월 또는 4월까지입니다. ...

# 질문: 북태평양 기단과 오호츠크해 기단이 만나 한반도에 머무르는 기간은?
# 소요 시간: 0.13s
# 답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 겨울철인 11월부터 3월 또는 4월까지입니다. ...

# 질문: 국내에 북태평양 기단과 오호츠크해 기단이 함께 머무르는 기간은?
# 소요 시간: 0.11s
# 답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 겨울철인 11월부터 3월 또는 4월까지입니다. ...

질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
소요 시간: 1.83s
답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 주로 가을부터 겨울까지이며, 일반적으로 10월부터 4월까지의 기간 동안 국내에 영향을 미칩니다.

질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
소요 시간: 0.00s
답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 주로 가을부터 겨울까지이며, 일반적으로 10월부터 4월까지의 기간 동안 국내에 영향을 미칩니다.

질문: 북태평양 기단과 오호츠크해 기단이 만나 한반도에 머무르는 기간은?
소요 시간: 0.27s
답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 주로 가을부터 겨울까지이며, 일반적으로 10월부터 4월까지의 기간 동안 국내에 영향을 미칩니다.

질문: 국내에 북태평양 기단과 오호츠크해 기단이 함께 머무리는 기간은?
소요 시간: 0.32s
답변: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은 주로 가을부터 겨울까지이며, 일반적으로 10월부터 4월까지의 기간 동안 국내에 영향을 미칩니다.



#### 데이터 검증

```
VsCode에서 실습